# Ridge regression on James's .pt files (Colab)

**Question:** does our baseline (embeddings → StandardScaler + PCA → ridge) still work when the data comes in James's PyTorch tensor format, on a different breast dataset?

**Why this notebook:** James's pipeline exports embeddings and expression as `.pt` (PyTorch) files, the format the learned model (task 2) will use. This notebook teaches our ridge code to read those files, so the same baseline runs on them. It's the same pipeline as `ridge_on_pca.ipynb`; only the loading step changes.

**The files** (breast, human, **Visium HD**; 500 spots sampled at random across all subjects)

| File | Shape | Meaning |
|---|---|---|
| `train_embeddings.pt` | (500, 4608) | spots × GenBio-PathFM embedding numbers → **X** (input) |
| `train_genes2.pt` | (500, 18085, 2) | spots × genes × [gene index, raw count] → **y** = `[:, :, 1]` (target) |

Row *i* is the same spot in both files (confirmed by James). "train" is just the file name, not a train/test split: we split the 500 spots ourselves.

**What this notebook does**
1. Connects Google Drive and loads the two `.pt` files with `torch.load`, then converts them to numpy arrays.
2. Turns the 3D gene tensor into a spots × genes count table and applies log1p.
3. Random **spot** split, 80/20 (400 train / 100 test).
4. Picks the top 50 most variable genes from the **training spots only**.
5. StandardScaler + PCA (fit on train only). PCA can't make more PCs than training spots, so 400 spots → at most 400 PCs instead of HEST's 256-from-thousands.
6. RidgeCV (λ chosen per gene by leave-one-out on the training spots) → **RMSE, NRMSE, PCC** on the test spots, next to the **mean predictor** and **ridge with no PCA**.
7. Plots, then saves the results to Drive.

**Limits of this data**
- **Spot split only.** The files have no patient or section IDs, so a slide or patient split isn't possible; scores are inflated by spatial leakage.
- **No gene names yet**, only indices 0–18,084, so genes show as `gene_<index>` until James sends the name list (set `GENE_NAMES_PATH`).
- **Small:** 500 spots. With 400 training spots and 4,608 numbers per spot, PCA with all PCs and no PCA give the same ridge result.

**Where it runs:** Google Colab, **CPU** runtime is enough.

**Before you run it:** upload both `.pt` files to a Google Drive folder (default: `/content/drive/MyDrive/BDD_BJMJ/pt_files`, set in `PT_FOLDER`).

**What it saves** (in `PT_FOLDER/ridge_results/`)

| File | Contents |
|---|---|
| `ridge_scores.csv` | RMSE, NRMSE, PCC per gene for the mean predictor, ridge with PCA, ridge without PCA, and ridge on the training spots |
| `ridge_predictions.npz` | Test true vs. predicted values, gene names and indices, train/test spot indices, chosen λ per gene |
| `pcc_per_gene.png`, `rmse_vs_baseline.png`, `true_vs_pred_<gene>.png` | Plots |

## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
import warnings
warnings.filterwarnings("ignore", message="Mean of empty slice")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Settings

- `PT_FOLDER`: the Drive folder with the two `.pt` files
- `GENE_NAMES_PATH`: optional text file with one gene name per line (18,085 lines). Leave `None` until James sends it; genes are then called `gene_<index>`
- `N_PCS`: number of PCs (HEST uses 256; capped by the number of training spots)

In [ ]:
PT_FOLDER = "/content/drive/MyDrive/BDD_BJMJ/pt_files"
EMBEDDINGS_FILE = "train_embeddings.pt"
GENES_FILE = "train_genes2.pt"
GENE_NAMES_PATH = None

N_GENES = 50
N_PCS = 256
TEST_FRACTION = 0.2
SEED = 0
ALPHAS = np.logspace(-2, 5, 30)
MIN_TEST_SD = 0.05

OUT_FOLDER = os.path.join(PT_FOLDER, "ridge_results")
os.makedirs(OUT_FOLDER, exist_ok=True)

## 2. Load the .pt files

`torch.load` opens each file as a tensor; `.numpy()` turns it into a normal numpy array so the rest of the code is the same as before.

In [ ]:
embeddings_tensor = torch.load(os.path.join(PT_FOLDER, EMBEDDINGS_FILE), map_location="cpu")
genes_tensor = torch.load(os.path.join(PT_FOLDER, GENES_FILE), map_location="cpu")

embeddings = embeddings_tensor.float().numpy()
genes = genes_tensor.float().numpy()

print("Embeddings:", embeddings.shape)
print("Genes:", genes.shape)

### Line up the two files on the spot axis

Both files must have the same number of spots on their first axis (500). Row *i* in one file is row *i* in the other.

In [ ]:
assert embeddings.shape[0] == genes.shape[0], "The two files have a different number of spots"
n_spots = embeddings.shape[0]
print("Spots in both files:", n_spots)

### Turn the 3D gene tensor into a count table

The last axis holds two values per gene: `[:, :, 0]` = gene index, `[:, :, 1]` = raw count.
The index is the same in every row, so we keep it once as the gene label and use the counts as y.

In [ ]:
if genes.ndim == 3 and genes.shape[2] == 2:
    gene_index = genes[0, :, 0].astype(int)
    assert np.all(genes[:, :, 0] == gene_index), "Gene index is not the same in every row"
    counts = genes[:, :, 1]
elif genes.ndim == 2:
    gene_index = np.arange(genes.shape[1])
    counts = genes
else:
    raise ValueError(f"Unexpected gene tensor shape: {genes.shape}")

if GENE_NAMES_PATH is not None:
    with open(GENE_NAMES_PATH) as f:
        all_gene_names = np.array([line.strip() for line in f])
    all_gene_names = all_gene_names[gene_index]
else:
    all_gene_names = np.array([f"gene_{i}" for i in gene_index])

print("Count table:", counts.shape, "(spots x genes)")
print("Share of zero counts:", round(float((counts == 0).mean()), 3))
print("Total counts per spot (first 5):", counts.sum(axis=1)[:5])

### Flatten the embeddings if needed

Ridge needs one row per spot. If the embedding tensor has extra axes (e.g. spots × neighbours × 4608), they are flattened into one long row. Our file is already 2D, so nothing changes.

In [ ]:
X_all = embeddings.reshape(n_spots, -1)
log_expr = np.log1p(counts)
print("X:", X_all.shape)
print("log1p expression:", log_expr.shape)

## 3. Random spot split, then top 50 genes from the training spots only

In [ ]:
spot_index = np.arange(n_spots)
train_idx, test_idx = train_test_split(spot_index, test_size=TEST_FRACTION, random_state=SEED)

gene_variance = log_expr[train_idx].var(axis=0)
top_gene_idx = np.argsort(gene_variance)[::-1][:N_GENES]
gene_names = all_gene_names[top_gene_idx]

X_train = X_all[train_idx]
X_test = X_all[test_idx]
y_train = log_expr[train_idx][:, top_gene_idx]
y_test = log_expr[test_idx][:, top_gene_idx]

print("Train spots:", len(train_idx))
print("Test spots:", len(test_idx))
print("Top 10 genes:", [str(name) for name in gene_names[:10]])

## 4. StandardScaler + PCA (fit on train only)

PCA can't have more PCs than training spots, so with 400 training spots we get at most 400 PCs.

In [ ]:
scaler = StandardScaler()
Z_train = scaler.fit_transform(X_train)
Z_test = scaler.transform(X_test)

n_components = min(N_PCS, Z_train.shape[0], Z_train.shape[1])
pca = PCA(n_components=n_components, random_state=SEED)
P_train = pca.fit_transform(Z_train)
P_test = pca.transform(Z_test)

rebuilt = pca.inverse_transform(P_test)
recon_r2 = 1 - np.sum((Z_test - rebuilt) ** 2) / np.sum((Z_test - Z_test.mean(axis=0)) ** 2)

print("PCs:", n_components)
print(f"Variance kept (train): {pca.explained_variance_ratio_.sum() * 100:.1f}%")
print(f"Reconstruction R2 (test): {recon_r2:.3f}")

## 5. Ridge regression, the mean predictor, and no PCA

- **Ridge**: one equation per gene, λ chosen per gene by leave-one-out cross-validation on the training spots
- **Mean predictor**: each gene's training average at every test spot (the floor)
- **No PCA**: ridge on all 4,608 scaled numbers. With fewer training spots than embedding numbers, this should match PCA with all PCs

In [ ]:
ridge_pca = RidgeCV(alphas=ALPHAS, alpha_per_target=True)
ridge_pca.fit(P_train, y_train)
y_pred_pca = ridge_pca.predict(P_test)
y_pred_pca_train = ridge_pca.predict(P_train)

ridge_nopca = RidgeCV(alphas=ALPHAS, alpha_per_target=True)
ridge_nopca.fit(Z_train, y_train)
y_pred_nopca = ridge_nopca.predict(Z_test)

y_pred_mean = np.tile(y_train.mean(axis=0), (len(y_test), 1))

print("Chosen alpha (PCA ridge), median over genes:", round(float(np.median(ridge_pca.alpha_)), 2))

## 6. Metrics, one gene at a time

- **RMSE**: how far off the values are (log1p units), lower is better
- **NRMSE** = RMSE ÷ SD of the gene's true test values; 1 ≈ no better than guessing the average
- **PCC**: does the prediction go up where the gene really goes up? Higher is better
- Genes whose test SD is below `MIN_TEST_SD` are skipped for NRMSE and PCC

In [ ]:
def rmse_one_gene(true_values, predicted_values):
    errors = true_values - predicted_values
    return np.sqrt(np.mean(errors ** 2))


def pcc_one_gene(true_values, predicted_values):
    if np.std(true_values) < 1e-6 or np.std(predicted_values) < 1e-6:
        return np.nan
    return np.corrcoef(true_values, predicted_values)[0, 1]


def score_all_genes(y_true, y_pred, model_name):
    rows = []
    for g in range(y_true.shape[1]):
        true_values = y_true[:, g]
        predicted_values = y_pred[:, g]
        rmse = rmse_one_gene(true_values, predicted_values)
        sd = np.std(true_values)
        if sd < MIN_TEST_SD:
            nrmse = np.nan
            pcc = np.nan
        else:
            nrmse = rmse / sd
            pcc = pcc_one_gene(true_values, predicted_values)
        rows.append({"model": model_name, "gene": gene_names[g], "test_sd": sd,
                     "RMSE": rmse, "NRMSE": nrmse, "PCC": pcc})
    return pd.DataFrame(rows)

In [ ]:
scores_mean = score_all_genes(y_test, y_pred_mean, "mean predictor")
scores_pca = score_all_genes(y_test, y_pred_pca, f"ridge, PCA {n_components}")
scores_nopca = score_all_genes(y_test, y_pred_nopca, "ridge, no PCA")
scores_train = score_all_genes(y_train, y_pred_pca_train, f"ridge, PCA {n_components} (train)")

all_scores = pd.concat([scores_mean, scores_pca, scores_nopca, scores_train])

print("Genes skipped for NRMSE/PCC (test SD too small):", int(scores_pca["NRMSE"].isna().sum()), "/", N_GENES)
print()
summary = all_scores.groupby("model", sort=False)[["RMSE", "NRMSE", "PCC"]].mean()
print(summary.round(3).to_string())

In [ ]:
ridge_table = scores_pca.sort_values("PCC", ascending=False)
print(ridge_table.round(3).to_string(index=False))

## 7. Plots

In [ ]:
plot_table = scores_pca.dropna(subset=["PCC"]).sort_values("PCC", ascending=False)

plt.figure(figsize=(14, 4))
plt.bar(plot_table["gene"], plot_table["PCC"])
plt.axhline(plot_table["PCC"].mean(), color="red", linestyle="--",
            label=f"mean PCC = {plot_table['PCC'].mean():.3f}")
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("PCC (test spots)")
plt.title(f"Visium HD breast (.pt files): ridge PCC per gene, PCA {n_components}")
plt.xticks(rotation=90)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_FOLDER, "pcc_per_gene.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, metric in zip(axes, ["RMSE", "NRMSE"]):
    x_values = scores_mean[metric]
    y_values = scores_pca[metric]
    ax.scatter(x_values, y_values, s=15)
    low = np.nanmin([x_values.min(), y_values.min()])
    high = np.nanmax([x_values.max(), y_values.max()])
    ax.plot([low, high], [low, high], color="grey", linestyle="--")
    ax.set_xlabel(f"Mean predictor {metric}")
    ax.set_ylabel(f"Ridge {metric}")
    ax.set_title(metric + " per gene (test spots)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_FOLDER, "rmse_vs_baseline.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
best_gene = plot_table["gene"].iloc[0]
g = list(gene_names).index(best_gene)

plt.figure(figsize=(5, 5))
plt.scatter(y_test[:, g], y_pred_pca[:, g], s=15)
low = min(y_test[:, g].min(), y_pred_pca[:, g].min())
high = max(y_test[:, g].max(), y_pred_pca[:, g].max())
plt.plot([low, high], [low, high], color="grey", linestyle="--")
best_pcc = scores_pca.loc[scores_pca["gene"] == best_gene, "PCC"].values[0]
plt.xlabel("True expression (log1p)")
plt.ylabel("Predicted expression (log1p)")
plt.title(f"{best_gene}: PCC = {best_pcc:.3f}")
plt.tight_layout()
plt.savefig(os.path.join(OUT_FOLDER, f"true_vs_pred_{best_gene}.png"), dpi=150, bbox_inches="tight")
plt.show()

## 8. Save the results to Drive

In [ ]:
all_scores.to_csv(os.path.join(OUT_FOLDER, "ridge_scores.csv"), index=False)
np.savez(
    os.path.join(OUT_FOLDER, "ridge_predictions.npz"),
    y_test=y_test, y_pred_ridge=y_pred_pca, y_pred_mean=y_pred_mean,
    gene_names=gene_names, gene_index=gene_index[top_gene_idx],
    train_idx=train_idx, test_idx=test_idx, chosen_alphas=ridge_pca.alpha_,
)
print("Saved to:", OUT_FOLDER)